# 01 · Synthetic India dataset — generation & exploratory analysis (offline)

**BAH 2026 PS3 · the offline data foundation for Objectives 1 & 2.**

Everything downstream (gap-fill → features → model → AQI; hotspots; transport) runs on a
small, deterministic **synthetic** India dataset produced by `aqi_india.sim` on the light
dependency set — no Earth Engine, no downloads, no GPU. This notebook generates that dataset
and explores its structure: the gridded satellite/met cube, the CPCB-like stations, and the
FIRMS-like fires, with maps and distributions. The on-disk schema matches `DEV_CONTRACT` §6.

In [ ]:
import sys, pathlib
# Make the src/ layout importable when running from the notebooks/ folder
# without an editable install. If aqi_india is already installed this is a no-op.
_repo = pathlib.Path.cwd()
for _ in range(4):
    if (_repo / 'src' / 'aqi_india').is_dir():
        sys.path.insert(0, str(_repo / 'src'))
        break
    _repo = _repo.parent
import aqi_india
print('aqi_india', aqi_india.__version__)

## 1. Generate the synthetic cube, fires and stations

The generators are pure functions in `aqi_india.sim.synthetic`:

- `make_grid(start_date, n_days, ...)` → an `xarray.Dataset` (`time, lat, lon`) with the 12
  data-vars of §6.1 (`aod`, `*_col` columns, `blh`, `rh`, `wind_u/v`, `t2m`, `ssrd`), with
  30–50% per-day **cloud gaps** (NaN) on the satellite column fields;
- `make_fires(grid, season=...)` → a FIRMS-like point GeoDataFrame (Punjab/Haryana stubble +
  a forest cluster) with `date, lat, lon, frp, sensor, h3_res7`;
- `inject_fire_hcho(grid, fires)` → adds a **downwind HCHO enhancement** so the fire→HCHO link
  is real and locatable;
- `make_stations(grid, n=...)` → CPCB-like station-days whose surface pollutants are a known
  nonlinear function of the co-located columns + BLH + RH.

In [ ]:
from aqi_india.sim import synthetic as sim
from aqi_india.sim.synthetic import GRID_VARS, COLUMN_VARS, SURFACE_POLLUTANTS

SEED = 42
grid = sim.make_grid('2023-10-01', n_days=60, res=0.25, seed=SEED)
fires = sim.make_fires(grid, season='oct_nov', seed=SEED)
grid = sim.inject_fire_hcho(grid, fires)            # downwind HCHO signal
stations = sim.make_stations(grid, n=120, seed=SEED)

print('grid dims      :', dict(grid.sizes))
print('grid data_vars :', list(grid.data_vars))
print('fires          :', len(fires), 'detections', sorted(fires['sensor'].unique()))
print('stations       :', stations['station_id'].nunique(), 'stations,', len(stations), 'station-days')

## 2. The gridded cube — columns, met and cloud gaps

Below: variable metadata, then the per-variable missing fraction. Only the satellite
**column** variables (`COLUMN_VARS`) carry cloud gaps; the reanalysis-style met fields are
gap-free by construction. This is exactly the missingness the gap-fill stage must close
(see notebook 03).

In [ ]:
import numpy as np
import pandas as pd

meta = pd.DataFrame({
    'units': {v: grid[v].attrs.get('units', '') for v in GRID_VARS},
    'long_name': {v: grid[v].attrs.get('long_name', '') for v in GRID_VARS},
    'pct_missing': {v: 100 * float(np.isnan(grid[v].values).mean()) for v in GRID_VARS},
})
meta.loc[list(GRID_VARS)]

### Map the key fields on one day

The IGP pollution gradient (high over the northern plains, low over the peninsula/oceans) is
visible in AOD and the NO2/HCHO columns; the speckled white holes on the column panels are the
synthetic cloud mask.

In [ ]:
import matplotlib.pyplot as plt

day = 40  # a post-monsoon day inside the burning window
panel_vars = ['aod', 'no2_col', 'hcho_col', 'blh', 'rh', 't2m']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, v in zip(axes.ravel(), panel_vars):
    grid[v].isel(time=day).plot(ax=ax, robust=True, cmap='turbo')
    ax.set_title(f"{v}  ({grid[v].attrs.get('units','')})")
fig.suptitle(f'Synthetic India cube — day index {day} ({str(grid.time.values[day])[:10]})', y=1.02)
plt.tight_layout(); plt.show()

### Wind field (the IGP transport corridor)

Prevailing north-west → south-east flow drives the fire→HCHO transport analysed in notebook 05.

In [ ]:
u = grid['wind_u'].isel(time=day).values
v = grid['wind_v'].isel(time=day).values
lon = grid['lon'].values; lat = grid['lat'].values
speed = np.hypot(u, v)
fig, ax = plt.subplots(figsize=(8, 7))
s = ax.streamplot(lon, lat, u, v, color=speed, cmap='cool', density=1.2)
fig.colorbar(s.lines, ax=ax, label='wind speed (m/s)')
ax.set_title('10 m wind — prevailing NW->SE flow'); ax.set_xlabel('lon'); ax.set_ylabel('lat')
plt.show()

## 3. Stations (CPCB surrogate labels)

One row per station-day, denser over the IGP / cities. The pollutant columns are in CPCB units
(CO in `mg/m3`, the rest `ug/m3`) and are the **labels** Objective-1 learns to predict.

In [ ]:
print('columns:', list(stations.columns))
stations.head()

In [ ]:
# Region breakdown + per-pollutant summary statistics.
display(stations['region'].value_counts().rename('n_station_days'))
stations[list(SURFACE_POLLUTANTS)].describe().T

### Station map and pollutant distributions

In [ ]:
from aqi_india.utils.geo import INDIA_BBOX

registry = stations.drop_duplicates('station_id')
fig, (axm, axh) = plt.subplots(1, 2, figsize=(14, 5.5))
sc = axm.scatter(registry['lon'], registry['lat'], c=registry['region'].astype('category').cat.codes,
                 cmap='tab10', s=30, edgecolor='k', linewidth=0.3)
axm.set_xlim(INDIA_BBOX[0], INDIA_BBOX[2]); axm.set_ylim(INDIA_BBOX[1], INDIA_BBOX[3])
axm.set_title(f'{len(registry)} synthetic CPCB stations (colour = region)')
axm.set_xlabel('lon'); axm.set_ylabel('lat'); axm.set_aspect('equal')
for p in ['pm25', 'pm10', 'no2', 'o3']:
    axh.hist(stations[p].dropna(), bins=40, histtype='step', label=p, linewidth=1.5)
axh.set_title('Surface pollutant distributions'); axh.set_xlabel('concentration'); axh.legend()
plt.tight_layout(); plt.show()

## 4. Fires (FIRMS surrogate)

FRP-weighted detections clustered in Punjab/Haryana (stubble) plus a forest cluster, weighted
into the Oct–Nov window. Notebook 05 turns these into burning **episodes** and a transport
attribution; here we just look at the spatial cluster, the FRP distribution and the daily count.

In [ ]:
from aqi_india.utils.geo import PUNJAB_HARYANA_BBOX

fig, (axf, axfrp, axts) = plt.subplots(1, 3, figsize=(17, 4.8))
axf.scatter(fires['lon'], fires['lat'], s=6 + fires['frp'] / 8, c=fires['frp'],
            cmap='hot_r', alpha=0.6)
axf.add_patch(plt.Rectangle((PUNJAB_HARYANA_BBOX[0], PUNJAB_HARYANA_BBOX[1]),
              PUNJAB_HARYANA_BBOX[2]-PUNJAB_HARYANA_BBOX[0],
              PUNJAB_HARYANA_BBOX[3]-PUNJAB_HARYANA_BBOX[1], fill=False, ec='navy', lw=1.5))
axf.set_xlim(INDIA_BBOX[0], INDIA_BBOX[2]); axf.set_ylim(INDIA_BBOX[1], INDIA_BBOX[3])
axf.set_title('Fire detections (size/colour = FRP)'); axf.set_aspect('equal')
axfrp.hist(fires['frp'], bins=40, color='orangered'); axfrp.set_title('FRP (MW) distribution')
axfrp.set_xlabel('FRP (MW)'); axfrp.set_yscale('log')
daily_ct = fires.groupby(fires['date'].dt.normalize()).size()
axts.plot(daily_ct.index, daily_ct.values, marker='.'); axts.set_title('Daily fire count')
axts.set_xlabel('date'); axts.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

## 5. (Optional) Write the dataset to disk in the DEV_CONTRACT layout

`generate_all` runs the whole chain and writes `grid.nc`, `stations.parquet`,
`fires.parquet` (+ geo variants) under `data/processed/`. The later notebooks can either call
the `sim` functions in-memory (as above) or load these files. Uncomment to materialise them.

In [ ]:
# from aqi_india.sim.synthetic import generate_all
# paths = generate_all('/home/user/bah2026-ps3', start_date='2023-10-01', n_days=60, seed=42)
# paths

## Summary

We generated and explored the synthetic India dataset: a 60-day, 0.25° gridded cube with
realistic IGP structure and cloud gaps, ~120 CPCB-like stations, and a Punjab/Haryana fire
cluster with a downwind HCHO signal. This is the offline substrate every other notebook builds on.